# 03 - Correlation Analysis
**DNA Gene Mapping Project - ML Phase V5 - Feature Selection**  
**Author:** Sharique Mohammad  
**Date:** February 2026

## Purpose
For every use case, compute the Pearson correlation matrix across all quality-passed
numeric features and flag pairs with absolute correlation above 0.95.
Also compute feature-target correlation to identify weak predictors.

## Output
`data/feature_selection/03_high_correlation_pairs.csv`
  - use_case, feature_a, feature_b, correlation, drop_feature (the one to remove)

`data/feature_selection/03_target_correlation.csv`
  - use_case, feature, target_correlation (sorted descending)

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from pathlib import Path
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

PROJECT_ROOT = Path().absolute().parent.parent
FS_DIR       = PROJECT_ROOT / 'data' / 'feature_selection'
FS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Setup complete")
print(f"Feature selection dir: {FS_DIR}")

## 2. Database Connection

In [ ]:
POSTGRES_HOST     = os.getenv("POSTGRES_HOST")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT")
POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)
print(f"Connected: {POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}")

## 3. Load Quality-Passed Feature Lists

In [ ]:
quality_df = pd.read_csv(FS_DIR / '02_feature_quality.csv')
kept_df    = quality_df[quality_df['keep'] == True]

# Build per-use-case feature list (quality-passed only)
uc_features = kept_df.groupby('use_case')['column'].apply(list).to_dict()
uc_table    = dict(zip(
    [r[0] for r in [
        ('UC01','clinical_ml_features'),('UC02','disease_ml_features'),
        ('UC03','pharmacogene_ml_features'),('UC04','variant_impact_ml_features'),
        ('UC05','structural_variant_ml_features'),('UC06','variant_drug_response_ml_features'),
        ('UC07','variant_cancer_ml_features'),('UC08','variant_population_ml_features'),
        ('UC09','population_frequency_ml_features'),('UC10','gene_pharmacogene_ml_features'),
        ('UC11','gene_expression_ml_features'),('UC12','gene_protein_family_ml_features'),
        ('UC13','gene_test_availability_ml_features'),('UC14','transcript_expression_ml_features'),
        ('UC15','cancer_variant_ml_features')
    ]],
    [r[1] for r in [
        ('UC01','clinical_ml_features'),('UC02','disease_ml_features'),
        ('UC03','pharmacogene_ml_features'),('UC04','variant_impact_ml_features'),
        ('UC05','structural_variant_ml_features'),('UC06','variant_drug_response_ml_features'),
        ('UC07','variant_cancer_ml_features'),('UC08','variant_population_ml_features'),
        ('UC09','population_frequency_ml_features'),('UC10','gene_pharmacogene_ml_features'),
        ('UC11','gene_expression_ml_features'),('UC12','gene_protein_family_ml_features'),
        ('UC13','gene_test_availability_ml_features'),('UC14','transcript_expression_ml_features'),
        ('UC15','cancer_variant_ml_features')
    ]]
))
uc_target = {
    'UC01':'target_is_pathogenic','UC02':'is_pathogenic','UC03':'is_pathogenic',
    'UC04':'is_high_impact','UC05':'sv_classification',
    'UC06':'is_actionable_pharmacogene_variant','UC07':'is_driver_candidate',
    'UC08':'is_carrier_screening_candidate','UC09':'is_clinically_actionable_rare_variant',
    'UC10':'is_high_priority_pharmacogene','UC11':'is_clinically_relevant_expression',
    'UC12':'is_high_value_protein_family','UC13':'is_high_priority_test_gene',
    'UC14':'is_clinically_relevant_expression','UC15':'gene_cancer_role'
}
uc_pct = {
    'UC01':10,'UC02':10,'UC03':10,'UC04':10,'UC05':100,
    'UC06':10,'UC07':10,'UC08':100,'UC09':100,'UC10':100,
    'UC11':100,'UC12':100,'UC13':100,'UC14':100,'UC15':10
}

print(f"Quality-passed feature lists loaded for {len(uc_features)} use cases")
for uc, feats in uc_features.items():
    print(f"  {uc}: {len(feats)} features")

## 4. Correlation Analysis Loop

In [ ]:
CORR_THRESHOLD = 0.95

high_corr_rows   = []
target_corr_rows = []

for uc, table in uc_table.items():
    feats  = uc_features.get(uc, [])
    target = uc_target[uc]
    pct    = uc_pct[uc]

    print(f"--- {uc} | {table} | {len(feats)} features ---")

    if len(feats) < 2:
        print("  Skipped: fewer than 2 features"); print(); continue

    df = pd.read_sql(
        f"SELECT * FROM gold.{table}" if pct == 100
        else f"SELECT * FROM gold.{table} TABLESAMPLE SYSTEM ({pct})",
        engine
    )

    # Bring all candidate features to numeric (booleans -> 0/1)
    numeric_df = pd.DataFrame()
    for col in feats:
        if col not in df.columns:
            continue
        s = df[col].astype(str).str.lower().map({'true':'1','false':'0'}).where(
            df[col].astype(str).str.lower().isin(['true','false']), other=df[col].astype(str))
        numeric_df[col] = pd.to_numeric(s, errors='coerce')

    corr_matrix = numeric_df.corr().abs()

    # Find high-correlation pairs
    cols = list(corr_matrix.columns)
    to_drop = set()
    pair_count = 0
    for i in range(len(cols)):
        if cols[i] in to_drop:
            continue
        for j in range(i+1, len(cols)):
            if cols[j] in to_drop:
                continue
            val = corr_matrix.iloc[i, j]
            if val > CORR_THRESHOLD:
                # Drop the one with lower mean absolute correlation
                mean_i = corr_matrix[cols[i]].mean()
                mean_j = corr_matrix[cols[j]].mean()
                drop   = cols[i] if mean_i < mean_j else cols[j]
                keep   = cols[j] if drop == cols[i] else cols[i]
                to_drop.add(drop)
                pair_count += 1
                high_corr_rows.append({
                    'use_case': uc, 'feature_a': cols[i], 'feature_b': cols[j],
                    'correlation': round(val, 4), 'drop_feature': drop, 'keep_feature': keep
                })

    print(f"  High-corr pairs (>{CORR_THRESHOLD}): {pair_count}  features to drop: {len(to_drop)}")

    # Target correlation (binary only)
    if target in df.columns and uc_target[uc] != 'gene_cancer_role' and uc_target[uc] != 'sv_classification':
        tgt = df[target].astype(str).str.lower().map({'true': 1, 'false': 0})
        tgt = pd.to_numeric(tgt, errors='coerce')
        for col in feats:
            if col in numeric_df.columns:
                r = numeric_df[col].corr(tgt)
                target_corr_rows.append({
                    'use_case': uc, 'feature': col,
                    'target_correlation': round(r, 4) if pd.notna(r) else 0
                })

    print()

## 5. Save Results

In [ ]:
high_corr_df   = pd.DataFrame(high_corr_rows)
target_corr_df = pd.DataFrame(target_corr_rows)

high_corr_df.to_csv(FS_DIR / '03_high_correlation_pairs.csv', index=False)
target_corr_df.to_csv(FS_DIR / '03_target_correlation.csv', index=False)

print("Saved: 03_high_correlation_pairs.csv")
print("Saved: 03_target_correlation.csv")
print()
print(f"Total high-corr pairs found : {len(high_corr_df)}")
print(f"Unique features to drop     : {high_corr_df['drop_feature'].nunique() if len(high_corr_df) > 0 else 0}")
print()
if len(high_corr_df) > 0:
    print("High correlation pairs (sample):")
    print(high_corr_df.head(20).to_string(index=False))

## 6. Visualize - Heatmaps for Key Use Cases

In [ ]:
# Show heatmap for UC01, UC04, UC10 as representative examples
spotlight_ucs = ['UC01', 'UC04', 'UC10']

for uc in spotlight_ucs:
    table  = uc_table.get(uc)
    feats  = uc_features.get(uc, [])
    pct    = uc_pct[uc]

    df = pd.read_sql(
        f"SELECT * FROM gold.{table}" if pct == 100
        else f"SELECT * FROM gold.{table} TABLESAMPLE SYSTEM ({pct})",
        engine
    )
    numeric_df = pd.DataFrame()
    for col in feats:
        if col not in df.columns: continue
        s = df[col].astype(str).str.lower().map({'true':'1','false':'0'}).where(
            df[col].astype(str).str.lower().isin(['true','false']), other=df[col].astype(str))
        numeric_df[col] = pd.to_numeric(s, errors='coerce')

    # Limit to 25 cols for readability
    corr = numeric_df[numeric_df.columns[:25]].corr()

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=0.5, ax=ax, annot_kws={'size': 7},
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{uc} - Feature Correlation Matrix (first 25 features)', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(FS_DIR / f'03_heatmap_{uc.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: 03_heatmap_{uc.lower()}.png")

## 7. Top Feature-Target Correlations

In [ ]:
if len(target_corr_df) > 0:
    top_n = 15
    fig, axes = plt.subplots(3, 4, figsize=(20, 14))
    axes = axes.flatten()
    binary_ucs = [uc for uc in uc_table.keys() if uc not in ('UC05', 'UC15')]

    for i, uc in enumerate(binary_ucs[:12]):
        uc_corr = target_corr_df[target_corr_df['use_case'] == uc].copy()
        if len(uc_corr) == 0:
            axes[i].axis('off'); continue
        uc_corr['abs_corr'] = uc_corr['target_correlation'].abs()
        top = uc_corr.nlargest(top_n, 'abs_corr')
        colors = ['#e74c3c' if v >= 0 else '#3498db' for v in top['target_correlation']]
        axes[i].barh(range(len(top)), top['target_correlation'].values,
                     color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
        axes[i].set_yticks(range(len(top)))
        axes[i].set_yticklabels(top['feature'].tolist(), fontsize=7)
        axes[i].set_title(f'{uc}', fontsize=9, fontweight='bold')
        axes[i].axvline(0, color='black', linewidth=0.5)
        axes[i].grid(axis='x', alpha=0.3)

    for i in range(len(binary_ucs[:12]), len(axes)):
        axes[i].axis('off')

    plt.suptitle(f'Top {top_n} Feature-Target Correlations per Use Case', fontweight='bold', fontsize=11)
    plt.tight_layout()
    plt.savefig(FS_DIR / '03_target_correlations.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 03_target_correlations.png")

## 8. Summary

In [ ]:
print("=" * 55)
print("CORRELATION ANALYSIS COMPLETE")
print("=" * 55)
print(f"High-corr pairs found  : {len(high_corr_df)}")
drop_by_uc = high_corr_df.groupby('use_case')['drop_feature'].nunique() if len(high_corr_df) > 0 else pd.Series(dtype=int)
for uc, n in drop_by_uc.items():
    print(f"  {uc}: {n} features to drop")
print()
print("Next: 04_leakage_detection.ipynb")